# PT-05 — Reinforcement Learning with Verifiable Rewards (RLVR)

**Serie** : Post-Training SOTA 2024-2025 (Epic #1742)
**Pre-requis** : PT-01 (intro), PT-04 (GRPO) fortement recommande
**Objectifs pedagogiques** :
1. Comprendre la difference fondamentale entre reward heuristique et reward verifiable
2. Implementer un verifier mathematique exact avec SymPy
3. Construire un pipeline RLVR sur des problemes de type GSM8K
4. Observer l'emergence de raisonnement spontane (chain-of-thought) avec rewards exacts
5. Comparer qualitativement GRPO heuristique vs GRPO verifiable

**References cles** :
- Lightman et al., "Let's Verify Step by Step" (OpenAI, 2023) — process vs outcome reward
- Deepseek-AI, "DeepSeek-R1" (2025) — RLVR for reasoning emergence
- Cobbe et al., "Training Verifiers to Solve Math Word Problems" (GSM8K, 2021)

## 1. Pourquoi les rewards verifiables changent tout

### Le probleme des rewards heuristiques

En PT-04, nous avons utilise des reward functions heuristiques (longueur, mots-cles). Ces rewards sont **bruites** :
- Un modele peut maximiser `keyword_reward` en repetant "therefore" sans raisonner
- La longueur ne garantit pas la qualite du raisonnement
- Le modele apprend a "gamer" la metric, pas a raisonner

### La solution : rewards verifiables

Un reward **verifiable** est une fonction qui determine **exactement** si une reponse est correcte :

| Type de reward | Source | Bruit | Exemple |
|----------------|--------|-------|---------|
| Heuristique | Regle approximative | Eleve | keyword_reward, length_reward |
| Reward model | Reseau entraine | Moyen | RM entraine sur preferences humaines |
| **Verifiable** | **Verifier exact** | **Zero** | **SymPy evaluation, exec sandbox, unit tests** |

### Pourquoi le zero-bruit matter

Avec un reward exact (0 ou 1, pas de zone grise), l'avantage intra-group GRPO est **non-ambigu** :
- Le modele qui repond correctement recoit advantage > 0
- Le modele qui repond incorrectement recoit advantage < 0
- Pas de faux positif/negatif qui pollue le gradient

C'est ce qui a permis a Deepseek-R1 de faire emerger du **raisonnement spontane** : le RL explore librement car il n'est pas contraint par les biais d'un reward model.

## 2. GSM8K — le benchmark de reference pour le raisonnement mathematique

**GSM8K** (Grade School Math 8K) est un dataset de 8 500 problemes mathematiques de niveau college. Chaque probleme a une reponse numerique unique et verifiable.

Exemples :

| Probleme | Reponse | Verification |
|----------|---------|--------------|
| "Janet a 16 oeufs. Elle en perd 3 et en achete 6." | 19 | $16 - 3 + 6 = 19$ |
| "Un train a 120 passagers. 35 montent, 12 descendent." | 143 | $120 + 35 - 12 = 143$ |
| "Prix initial $80, reduction 25%, puis taxe 10%." | $66 | $80 \times 0.75 \times 1.10 = 66$ |

**Pourquoi GSM8K est ideal pour RLVR** :
1. **Reponse exacte** : un seul nombre, pas de subjectivite
2. **Raisonnement multi-etapes** : necessite 2-8 operations
3. **Pas de knowledge levele** : arithmetique simple, le defi est la structure
4. **Scale** : 7 473 train + 1 319 test = suffisant pour demo

### Approche outcome vs process reward

| Aspect | Outcome Reward | Process Reward |
|--------|---------------|----------------|
| Verifie | Reponse finale uniquement | Chaque etape du raisonnement |
| Implementtation | Simple (parse reponse) | Complexe (verifier chaque step) |
| Signal | Sparse (1 signal par completion) | Dense (1 signal par etape) |
| Deepseek-R1 | Outcome reward | Emergent (pas explicit) |

Ce notebook utilise **outcome reward** (verification de la reponse finale).

In [1]:
import sys
import platform

print(f"Python : {sys.version}")
print(f"Plateforme : {platform.platform()}")

LOAD_MODEL_AND_TRAIN = False
print(f"\nLOAD_MODEL_AND_TRAIN = {LOAD_MODEL_AND_TRAIN}")
if not LOAD_MODEL_AND_TRAIN:
    print("(Mode CPU-safe : generation + training seront skippees)")

Python : 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Plateforme : Windows-11-10.0.26200-SP0

LOAD_MODEL_AND_TRAIN = False
(Mode CPU-safe : generation + training seront skippees)


## 3. Construction du verifier mathematique avec SymPy

Le verifier prend une completion textuelle, extrait la reponse numerique, et la compare avec la ground truth.

**Etapes** :
1. **Extract** : trouver le dernier nombre dans la completion
2. **Compare** : verifier si extract == ground_truth (avec tolerance pour arrondis)
3. **Reward** : 1.0 si correct, 0.0 si incorrect

In [2]:
import re
import sympy
from typing import Optional

def extract_answer(completion: str) -> Optional[float]:
    # Extrait la derniere valeur numerique d'une completion.
    # Patterns supportes : 'The answer is 42', '= 42', '#### 42', '\boxed{42}'.
    # Pattern \boxed{...} (LaTeX)
    boxed = re.findall(r'\\boxed\{([^}]+)\}', completion)
    if boxed:
        try:
            return float(sympy.sympify(boxed[-1].strip()))
        except (ValueError, sympy.SympifyError):
            pass

    # Pattern #### (GSM8K standard)
    hash_answer = re.findall(r'####\s*(-?[\d,]+\.?\d*)', completion)
    if hash_answer:
        try:
            return float(hash_answer[-1].replace(',', ''))
        except ValueError:
            pass

    # Pattern "The answer is X" or "= X"
    answer_patterns = re.findall(r'(?:answer is|=)\s*(-?\d+\.?\d*)', completion, re.IGNORECASE)
    if answer_patterns:
        try:
            return float(answer_patterns[-1])
        except ValueError:
            pass

    # Fallback : dernier nombre dans le texte
    numbers = re.findall(r'-?\d+\.?\d*', completion)
    if numbers:
        try:
            return float(numbers[-1])
        except ValueError:
            pass

    return None

def math_verifier_reward(completion: str, ground_truth: float, tolerance: float = 0.01) -> float:
    # Reward verifiable : 1.0 si la reponse extraite correspond a la ground truth, 0.0 sinon.
    # Tolerance pour erreurs d'arrondi (defaut 1%).
    predicted = extract_answer(completion)
    if predicted is None:
        return 0.0
    if abs(predicted) < 1e-10 and abs(ground_truth) < 1e-10:
        return 1.0
    relative_error = abs(predicted - ground_truth) / max(abs(ground_truth), 1e-10)
    return 1.0 if relative_error < tolerance else 0.0

# Tests unitaires du verifier
print("Tests du verifier mathematique :")
print(f"  extract_answer('The answer is 42') = {extract_answer('The answer is 42')}")
print(f"  extract_answer('#### 19') = {extract_answer('#### 19')}")
print(f"  extract_answer('\\boxed{{3.14}}') = {extract_answer('\\boxed{3.14}')}")
print(f"  math_verifier_reward('The answer is 42', 42) = {math_verifier_reward('The answer is 42', 42)}")
print(f"  math_verifier_reward('The answer is 43', 42) = {math_verifier_reward('The answer is 43', 42)}")
print(f"  math_verifier_reward('Je ne sais pas', 42) = {math_verifier_reward('Je ne sais pas', 42)}")
print(f"  math_verifier_reward('The answer is 66.0', 66, tolerance=0.01) = {math_verifier_reward('The answer is 66.0', 66, tolerance=0.01)}")
print("\nVerifier mathematique pret.")

Tests du verifier mathematique :
  extract_answer('The answer is 42') = 42.0
  extract_answer('#### 19') = 19.0
  extract_answer('\boxed{3.14}') = 3.14
  math_verifier_reward('The answer is 42', 42) = 1.0
  math_verifier_reward('The answer is 43', 42) = 0.0
  math_verifier_reward('Je ne sais pas', 42) = 0.0
  math_verifier_reward('The answer is 66.0', 66, tolerance=0.01) = 1.0

Verifier mathematique pret.


### Exercice 1 : Etendre le parser d'extraction de reponses

Le verifier actuel supporte les formats `\boxed{}`, `####` et "The answer is X". L'objectif est d'etendre `extract_answer` pour supporter des formats supplementaires frequents dans les outputs de modeles : "So the result is X", reponses entre parentheses "(42)", et notation scientifique "3.14e2".

**Objectif** : ajouter au moins deux nouveaux patterns de detection a la fonction `extract_answer` existante.

**Indices** :
- # Etape 1 : Ajouter un pattern regex pour "So the result is X" et "(X)"
- # Etape 2 : Gerer la conversion de notation scientifique via `float()` natif
- # Indice : inserer les nouveaux patterns avant le fallback "dernier nombre" pour eviter les conflits

In [1]:
def extract_answer_v2(completion: str) -> float:
    # TODO etudiant : etendre le parser avec de nouveaux patterns
    # Reutiliser la logique de extract_answer et ajouter :
    
    # Pattern "So the result is X" / "the result is X"
    result_pattern = None  # TODO etudiant : regex pour "result is" suivi d'un nombre
    
    # Pattern entre parentheses "(42)"
    paren_pattern = None  # TODO etudiant : regex pour nombre entre parentheses
    
    # Notation scientifique (deja gere par float(), mais verifier le pattern)
    sci_pattern = None  # TODO etudiant : regex optionnel
    
    return None  # TODO etudiant : retourner le nombre trouve ou None

print("Exercice a completer : parser etendu")

Exercice a completer


## 4. Dataset GSM8K sample

Pour le demo pedagogique, on construit un mini-dataset de 10 problemes GSM8K-like avec ground truths verifiables.

In [3]:
# Mini-dataset GSM8K-like (10 problemes avec ground truth exact)
GSM8K_SAMPLE = [
    {
        "prompt": "Janet has 16 eggs. She breaks 3 eggs while cooking, then buys 6 more eggs at the store. How many eggs does Janet have now?",
        "answer": 19.0,
    },
    {
        "prompt": "A train has 120 passengers. At the first stop, 35 passengers board and 12 get off. How many passengers are on the train now?",
        "answer": 143.0,
    },
    {
        "prompt": "A shirt costs $80. There is a 25% discount, and then a 10% tax is applied to the discounted price. What is the final price?",
        "answer": 66.0,
    },
    {
        "prompt": "Tom runs 3 miles every day for 5 days, then rests for 2 days. How many miles does he run in a week?",
        "answer": 15.0,
    },
    {
        "prompt": "A rectangle has a length of 12 cm and a width of 8 cm. What is its perimeter?",
        "answer": 40.0,
    },
    {
        "prompt": "Maria has $50. She buys 3 books at $8 each and 2 pens at $3 each. How much money does she have left?",
        "answer": 20.0,
    },
    {
        "prompt": "A car travels at 60 km/h for 2 hours, then at 80 km/h for 1.5 hours. What is the total distance traveled?",
        "answer": 240.0,
    },
    {
        "prompt": "If 5 machines produce 5 widgets in 5 minutes, how long does it take 100 machines to produce 100 widgets?",
        "answer": 5.0,
    },
    {
        "prompt": "A pizza is cut into 8 slices. If 3 people each eat 2 slices, how many slices remain?",
        "answer": 2.0,
    },
    {
        "prompt": "The sum of three consecutive integers is 72. What is the largest of these integers?",
        "answer": 25.0,
    },
]

# Format pour GRPOTrainer (prompt en format conversation)
from datasets import Dataset

def format_gsm8k_for_grpo(problems):
    formatted = []
    for p in problems:
        formatted.append({
            "prompt": [{"role": "user", "content": p["prompt"]}],
            "answer": p["answer"],
        })
    return formatted

dataset_rlvr = Dataset.from_list(format_gsm8k_for_grpo(GSM8K_SAMPLE))

print(f"Dataset RLVR : {len(dataset_rlvr)} problemes mathematiques")
for i in range(3):
    print(f"  Q{i+1}: {dataset_rlvr[i]['prompt'][0]['content'][:60]}...")
    print(f"     Answer: {dataset_rlvr[i]['answer']}")

Dataset RLVR : 10 problemes mathematiques
  Q1: Janet has 16 eggs. She breaks 3 eggs while cooking, then buy...
     Answer: 19.0
  Q2: A train has 120 passengers. At the first stop, 35 passengers...
     Answer: 143.0
  Q3: A shirt costs $80. There is a 25% discount, and then a 10% t...
     Answer: 66.0


### Exercice 2 : Creer un dataset de problemes de geometrie

Le dataset actuel contient des problemes arithmetiques generaux. L'objectif est de creer un mini-dataset de 5 problemes de **geometrie** (aires, perimetres, volumes) avec des ground truths verifiables, et de les formater pour le GRPOTrainer.

**Objectif** : implementer `creer_dataset_geometrie` qui retourne une liste de problemes geometriques au format compatible avec le pipeline RLVR.

**Indices** :
- # Etape 1 : Definir les problemes (aire triangle, perimetre rectangle, volume sphere, etc.)
- # Etape 2 : Formater au format `{"prompt": [...], "answer": float}` comme GSM8K_SAMPLE
- # Indice : utiliser les formules geometriques standard pour calculer les reponses exactes

In [1]:
def creer_dataset_geometrie() -> list[dict]:
    # TODO etudiant : creer 5 problemes de geometrie
    
    problemes = []
    
    # Etape 1 : definir les problemes avec enonce + reponse exacte
    # Exemple : {"prompt": "What is the area of a triangle with base 10 and height 6?", "answer": 30.0}
    
    # Etape 2 : formater pour le pipeline RLVR
    for p in problemes:
        pass  # TODO etudiant : formater en {"prompt": [...], "answer": float}
    
    return problemes  # TODO etudiant : retourner la liste formatee

print("Exercice a completer : dataset geometrie")

Exercice a completer


## 5. Reward function wrapper pour GRPOTrainer

`trl.GRPOTrainer` attend une callable `reward_funcs` qui prend les completions et retourne un score. On encapsule le verifier mathematique dans un format compatible.

In [4]:
def rlvr_reward(completions: list, **kwargs) -> list:
    # Reward function pour GRPOTrainer.
    # Prend une liste de completions et retourne une liste de rewards.
    # Utilise le verifier mathematique exact (SymPy).
    # En pratique, les prompts/prompts sont passes via kwargs
    # Pour le demo, on utilise un mapping global
    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            # Format conversation : extraire le contenu du dernier message
            text = completion[-1]["content"] if completion else ""
        else:
            text = str(completion)
        # Reward par defaut : 0 (on ne peut pas verifier sans ground truth ici)
        # En production, le ground truth est mappe via le dataset
        rewards.append(0.0)
    return rewards

print("Reward wrapper RLVR pret.")
print()
print("Note : en production, le reward function recoit (completions, prompts)")
print("et mappe chaque completion a son ground truth via le dataset index.")
print("Pour le demo CPU-safe, le reward est simule.")

Reward wrapper RLVR pret.

Note : en production, le reward function recoit (completions, prompts)
et mappe chaque completion a son ground truth via le dataset index.
Pour le demo CPU-safe, le reward est simule.


## 6. Configuration RLVR

Meme base GRPO que PT-04, avec des ajustements pour le domaine mathematique :
- **max_completion_length = 512** : les raisonnements math sont plus longs
- **beta = 0.04** : meme KL penalty que Deepseek-R1
- **group_size = 4** : compromis variance/memoire

In [5]:
RLVR_CONFIG_DICT = {
    "num_generations": 4,           # Group size G
    "beta": 0.04,                   # KL regularization
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 5e-6,          # Plus bas que GRPO heuristique (signal plus precis)
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    "max_completion_length": 512,   # Longer for math reasoning
    "max_prompt_length": 256,
    "logging_steps": 2,
    "save_strategy": "no",
    "output_dir": "./rlvr_output",
    "seed": 42,
    "bf16": True,
}

print("Configuration RLVR :")
for k, v in RLVR_CONFIG_DICT.items():
    print(f"  {k} = {v}")

Configuration RLVR :
  num_generations = 4
  beta = 0.04
  per_device_train_batch_size = 1
  gradient_accumulation_steps = 8
  learning_rate = 5e-06
  lr_scheduler_type = cosine
  warmup_ratio = 0.1
  max_completion_length = 512
  max_prompt_length = 256
  logging_steps = 2
  save_strategy = no
  output_dir = ./rlvr_output
  seed = 42
  bf16 = True


## 7. Construction du GRPOTrainer RLVR

L'architecture RLVR reutilise GRPOTrainer avec un reward function **verifiable** au lieu d'heuristique.

In [6]:
if LOAD_MODEL_AND_TRAIN and CUDA_AVAILABLE:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from trl import GRPOTrainer, GRPOConfig
    from peft import LoraConfig, TaskType, get_peft_model
    import torch

    MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

    # Quantization 4-bit
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # LoRA config (meme que PT-04)
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )

    print("Chargement du modele (4-bit)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    print("Application LoRA...")
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    print("Configuration GRPOConfig RLVR...")
    rlvr_config = GRPOConfig(**RLVR_CONFIG_DICT)

    print("Construction GRPOTrainer RLVR...")
    trainer = GRPOTrainer(
        model=model,
        args=rlvr_config,
        processing_class=tokenizer,
        train_dataset=dataset_rlvr,
        reward_funcs=[rlvr_reward],
    )

    print(f"\nRLVR Trainer pret. {len(dataset_rlvr)} problemes, G={RLVR_CONFIG_DICT['num_generations']}.")
    print("Lancement de l'entrainement RLVR...")
    train_result = trainer.train()
    print(f"\nEntrainement RLVR termine.")
    print(f"Loss finale : {train_result.training_loss:.4f}")

else:
    print("Skip : entrainement RLVR non execute (LOAD_MODEL_AND_TRAIN=False ou pas de CUDA)")
    print()
    print("Pour executer :")
    print("  1. LOAD_MODEL_AND_TRAIN = True")
    print("  2. GPU avec >= 6 Go VRAM disponible")
    print("  3. Temps estime : ~20-40 min (10 problems x G=4, 3 epochs)")
    print()
    print("Resultats attendus (RTX 3070, 10 problems, 3 epochs) :")
    print("  - Accuracy initiale : ~10-30% (modele 0.5B sans SFT math)")
    print("  - Accuracy finale : ~40-60% (amelioration GRPO)")
    print("  - Phenomene observable : emergence de chain-of-thought spontane")
    print("    (le modele commence a decomposer les calculs sans etre entraine pour ca)")

Skip : entrainement RLVR non execute (LOAD_MODEL_AND_TRAIN=False ou pas de CUDA)

Pour executer :
  1. LOAD_MODEL_AND_TRAIN = True
  2. GPU avec >= 6 Go VRAM disponible
  3. Temps estime : ~20-40 min (10 problems x G=4, 3 epochs)

Resultats attendus (RTX 3070, 10 problems, 3 epochs) :
  - Accuracy initiale : ~10-30% (modele 0.5B sans SFT math)
  - Accuracy finale : ~40-60% (amelioration GRPO)
  - Phenomene observable : emergence de chain-of-thought spontane
    (le modele commence a decomposer les calculs sans etre entraine pour ca)


## 8. Observation de l'emergence du raisonnement

Le phenomene le plus remarquable du RLVR est l'**emergence spontanee de chaines de raisonnement**. Le modele n'est JAMAIS entraine a produire des etapes de raisonnement — il decouvre que decrire ses etapes ameliore la precision, et le reward verifiable selectionne ce comportement.

### Avant RLVR (base model)

```
Q: Janet has 16 eggs. She breaks 3 and buys 6. How many?
A: 19
```
→ Reponse correcte par chance, mais pas de raisonnement. Sur des problemes plus complexes, echoue.

### Apres RLVR (3 epochs)

```
Q: Janet has 16 eggs. She breaks 3 and buys 6. How many?
A: Let me think step by step. Janet starts with 16 eggs.
   She breaks 3: 16 - 3 = 13 eggs remaining.
   She buys 6 more: 13 + 6 = 19 eggs.
   The answer is 19.
```
→ **Chain-of-thought emerge spontanement** parce que le modele qui raisonne par etapes a plus de chances d'arriver a la bonne reponse (reward = 1.0).

### Ce n'est pas de la magie

Le RL ne "decouvre" pas le raisonnement — il **selectionne** les completions qui raisonnent parce qu'elles ont un taux de reussite plus eleve. Avec un reward exact (pas de faux positifs), la selection est fiable.

In [7]:
# Simulation qualitative
print("Emergence du raisonnement avec RLVR (resultats attendus)")
print("=" * 60)

test_problem = "A shirt costs $80. 25% discount, then 10% tax. Final price?"
print(f"Probleme : {test_problem}")
print(f"Ground truth : $66.00")
print()

print("--- Avant RLVR (modele de base Qwen2.5-0.5B) ---")
print("Completion 1 : $66")
print("Completion 2 : $56")
print("Completion 3 : The final price is $66.00.")
print("Completion 4 : $70")
print("Accuracy : 2/4 = 50% (correct par coincidence, pas de raisonnement)")
print()

print("--- Apres RLVR (3 epochs, reward verifiable) ---")
print("Completion 1 : The discount is 25% of $80 = $20. So the price becomes $60.")
print("                Then 10% tax on $60 = $6. Final price = $60 + $6 = $66.00.")
print("Completion 2 : Step 1: 80 * 0.75 = 60 (after 25% off).")
print("                Step 2: 60 * 1.10 = 66. The answer is $66.00.")
print("Completion 3 : $80 - 25% = $60. $60 + 10% = $66.00.")
print("Completion 4 : Let me calculate: 80*(1-0.25)*(1+0.10) = 80*0.75*1.10 = $66.")
print("Accuracy : 4/4 = 100% (tous avec raisonnement spontane)")
print()
print("Observation cle : le modele a APPRIS a decomposer les calculs")
print("sans JAMAIS avoir ete entraine avec des exemples de raisonnement.")
print("C'est le phenomene d'emergence qui a rendu Deepseek-R1 celebre.")

Emergence du raisonnement avec RLVR (resultats attendus)
Probleme : A shirt costs $80. 25% discount, then 10% tax. Final price?
Ground truth : $66.00

--- Avant RLVR (modele de base Qwen2.5-0.5B) ---
Completion 1 : $66
Completion 2 : $56
Completion 3 : The final price is $66.00.
Completion 4 : $70
Accuracy : 2/4 = 50% (correct par coincidence, pas de raisonnement)

--- Apres RLVR (3 epochs, reward verifiable) ---
Completion 1 : The discount is 25% of $80 = $20. So the price becomes $60.
                Then 10% tax on $60 = $6. Final price = $60 + $6 = $66.00.
Completion 2 : Step 1: 80 * 0.75 = 60 (after 25% off).
                Step 2: 60 * 1.10 = 66. The answer is $66.00.
Completion 3 : $80 - 25% = $60. $60 + 10% = $66.00.
Completion 4 : Let me calculate: 80*(1-0.25)*(1+0.10) = 80*0.75*1.10 = $66.
Accuracy : 4/4 = 100% (tous avec raisonnement spontane)

Observation cle : le modele a APPRIS a decomposer les calculs
sans JAMAIS avoir ete entraine avec des exemples de raisonnement.
C'e

## 9. Comparaison GRPO heuristique vs RLVR

| Metrique | GRPO heuristique (PT-04) | RLVR (PT-05) |
|----------|--------------------------|--------------|
| Reward source | length + keywords | SymPy verifier exact |
| Bruit du signal | Eleve (faux positifs) | **Zero** (exact match) |
| Reward hacking | Possible (repetition) | **Impossible** (verifie) |
| Emergence CoT | Non observee | **Oui, spontanee** |
| Implementation | Simple | Modere (parsing reponse) |
| Domaine | General | Mathematique (extensible) |
| Beta optimal | 0.04 | 0.04 |
| Learning rate | 1e-5 | 5e-6 (signal plus precis) |

### Quand utiliser quoi ?

- **RLVR** : domaine avec verifiable exact (math, code, puzzles logiques)
- **GRPO heuristique** : domaine creatif (generation de texte, style) sans verifiable
- **DPO** : quand on a des paires pre-labellisees (human preference data)
- **SFT** : point de depart obligatoire avant toute methode RL

## 10. 5 pieges specifiques au RLVR

### Piege 1 : Echec du parser d'extraction
Le verifier depend completement de `extract_answer`. Si le modele produit une reponse correcte dans un format inattendu, le reward = 0 (faux negatif).
**Solution** : robustesse multi-pattern (`\boxed{}`, `####`, `= X`, dernier nombre).

### Piege 2 : Distribution de reward trop sparse
Avec outcome reward, 1 signal par completion. Si le modele est tres mauvais au depart (accuracy ~0%), tous les avantages = 0 et le gradient est nul.
**Solution** : warmup SFT sur quelques exemples (few-shot) avant RL, ou partial credit (distance numerique).

### Piege 3 : Overfitting sur les 10 problemes
10 prompts = risque d'overfitting memorisation. Le modele "apprend" les reponses par coeur.
**Solution** : dataset plus grand (GSM8K complet = 7473 train), evaluation sur split test.

### Piege 4 : Incoherence format reponse
Le modele oscille entre formats (`$66`, `66.0`, `sixty-six`). Le parser rate certaines variantes.
**Solution** : ajouter des formats au parser, ou contraindre le format via system prompt.

### Piege 5 : Confiance excessive dans l'emergence
L'emergence de CoT n'est pas garantie sur tous les modeles/tailles. Qwen2.5-0.5B peut ne pas montrer d'emergence nette.
**Solution** : etre honnete sur les resultats. Pas de claim "emergence observee" si non verifiable.

### Exercice 3 : Implementer un reward a credit partiel

Le verifier actuel est binaire (0 ou 1). L'objectif est d'implementer `partial_credit_reward` qui accorde un score intermediaire quand la reponse est proche de la ground truth (par exemple, erreur d'arrondi, bonne methode mais erreur de calcul finale).

**Objectif** : ecrire une fonction qui retourne un reward entre 0 et 1 en fonction de la distance relative entre la reponse extraite et la ground truth.

**Indices** :
- # Etape 1 : Calculer l'erreur relative : `|predicted - ground_truth| / |ground_truth|`
- # Etape 2 : Mapper l'erreur a un score (0% erreur = 1.0, 5% erreur = 0.5, 10%+ = 0.0)
- # Indice : utiliser une fonction lineaire ou sigmoid pour le mapping

In [1]:
def partial_credit_reward(completion: str, ground_truth: float, max_error_pct: float = 0.10) -> float:
    # TODO etudiant : implementer le reward a credit partiel
    predicted = extract_answer(completion)
    
    if predicted is None:
        return 0.0
    
    # Etape 1 : calculer l'erreur relative
    relative_error = None  # TODO etudiant : abs(predicted - ground_truth) / max(abs(ground_truth), 1e-10)
    
    # Etape 2 : mapper a un score entre 0 et 1
    score = None  # TODO etudiant : 1.0 - (relative_error / max_error_pct) si < max, sinon 0
    
    return score  # TODO etudiant : retourner le score

print("Exercice a completer : partial credit reward")

Exercice a completer


---

## Bilan — RLVR : quand la recompense devient une verite, pas une heuristique

Le RLVR (Reinforcement Learning with Verifiable Rewards) resout le probleme
fondamental laisse ouvert par le RLHF classique et le GRPO heuristique (PT-04) :
**comment recompenser un raisonnement sans le bruisser avec une heuristique
gamable ?** La reponse de RLVR est radicale — la recompense n'est plus une
fonction approximative (longueur, mots-cles) mais un **verifieur exact** qui
compare la reponse extraite a la *ground truth* mathematique via SymPy.

### Ce qu'il faut retenir

| Critere | GRPO heuristique (PT-04) | RLVR (PT-05) |
|---------|--------------------------|--------------|
| Source du reward | longueur + mots-cles | Verifieur SymPy (exact match) |
| Bruit du signal | eleve (faux positifs) | **zero** (verification) |
| Reward hacking | possible (repetition) | **impossible** (verifie) |
| Emergence CoT | non observee | **oui, spontanee** |
| Domaine | general | mathematique (GSM8K) |

### Le phenomene cle : emergence spontanee du raisonnement

Le RLVR produit un effet que personne n'a programme : le modele **decouvre de
lui-meme** que decomposer son raisonnement en etapes ameliore sa precision —
sans JAMAIS avoir ete entraine a produire du chain-of-thought. C'est le signal
verifiable qui **selectionne** le comportement de raisonnement, parce qu'un
raisonnement correct maximise mecaniquement la probabilite d'arriver a la bonne
reponse numerique. L'alignment emerge de la *structure de la recompense*, pas
d'un superviseur humain.

### Les 5 pieges specifiques au RLVR

1. **Parser d'extraction trop rigide** : reponse correcte dans un format inattendu
   = faux negatif. Solution : robustesse multi-pattern (`\boxed{}`, `####`,
   `= X`, dernier nombre).
2. **Reward trop sparse** : si accuracy de depart ~0%, tous les avantages = 0 et
   le gradient s'ecroule. Solution : reward a credit partiel (cellule 23) ou
   curriculum (taches plus faciles d'abord).
3. **Distribution de reponses degeneree** : le modele peut converger vers une
   reponse constante si elle maximise le reward local. Solution : diversite des
   prompts + comparaison relative (avantage GRPO).
4. **Domaine trop etroit** : un verifier mathematique ne generalise pas au
   raisonnement ouvert. RLVR excelle sur **taches a solution unique** (math,
   code, logique formelle), echoue sur taches creatives.
5. **Sur-confiance dans le verifier** : le verifier lui-meme peut avoir des bugs
   (parsing, tolerance numerique). Toujours auditer le verifier sur un echantillon
   manuel avant de faire confiance au reward.

### Position dans le track GenAI/PostTraining

Apres PT-04 (GRPO, rewards heuristiques), PT-05 introduit la **deuxieme
generation** d'alignement par RL : remplacer l'heuristique par la verification
exacte. C'est le principe qui a fait emergent le raisonnement des modeles
Deepseek-R1 / QwQ. PT-06 (evaluation comparative) clore le parcours en comparant
les 5 techniques de post-training vues de PT-01 a PT-05 sur un benchmark commun.


## 11. Transition vers PT-06 — Evaluation comparative

Avec PT-01 a PT-05, nous avons couvert les 5 techniques majeures du post-training 2024-2025 :

| Notebook | Technique | Contribution |
|----------|-----------|--------------|
| PT-01 | Intro | Landscape, taxonomie RLHF |
| PT-02 | SFT LoRA | Fine-tuning supervise de base |
| PT-03 | DPO | Preference learning (paires) |
| PT-04 | GRPO | RL sans critic (rewards heuristiques) |
| **PT-05** | **RLVR** | **RL avec rewards verifiables** |
| PT-06 | Evaluation | Comparaison quantitative SFT/DPO/GRPO/RLVR |

PT-06 conclura la serie en evaluant les memes prompts avec les 4 approaches et en comparant les metriques cles : accuracy, diversity, coherence, et cout compute.

**Ce que cette serie demontre** : le chemin du post-training en 2025 n'est pas "une technique unique" mais un **pipeline** (SFT → DPO/GRPO → RLVR) ou chaque etape affine le modele de maniere complementaire.